<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_DeepONet_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# ============================================================================
# GENERIC-DeepONet PARAM-MATCHED ABLATION -- ONE SELF-CONTAINED COLAB CELL.
# Nothing else to run. Paste this whole thing in a single cell and execute.
# Part A = DeepONet + GENERIC-DeepONet classes and 1D generators (verbatim);
# Part C = trains both models at two matched parameter budgets (~58K and ~175K)
# on heat / advection / Burgers and prints a matched-budget table + LaTeX, and
# saves fig_deeponet_matched.{pdf,png}.
# Optional: set os.environ["GENERIC_FNO_FIGDIR"]=...  and AB_EPOCHS / AB_NX etc.
# BEFORE this cell.
# ============================================================================

# ==================== PART A: DEEPONET MODELS + GENERATORS (verbatim) ========
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict

# ============================================================================
# Data Generation (same 5 PDEs)
# ============================================================================

def generate_heat_data(n_samples=200, nx=64, nt=20, dt=0.01, nu=0.01):
    """Heat equation: du/dt = nu * d²u/dx². Purely dissipative."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    decay = torch.exp(-nu * k**2 * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)  # (nt+1, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'heat'

def generate_wave_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0):
    """Wave equation (1st order system): du/dt = c*dv/dx, dv/dt = c*du/dx.
    Purely reversible (Hamiltonian). We track u-component only."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    omega = c * k  # dispersion relation

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        v0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)
            # Give some initial velocity too
            amp_v = torch.randn(1).item() * 0.3
            v0 += amp_v * torch.cos(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        v_hat = torch.fft.rfft(v0)

        traj = [u0.clone()]
        for t in range(nt):
            # Exact solution: rotate in (u_hat, v_hat) space
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            u_new = cos_w * u_hat + 1j * sin_w * v_hat
            v_new = 1j * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'wave'

def generate_advection_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0, max_mode=6):
    """1D linear advection du/dt + c*du/dx = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly => thermodynamically-consistent model drives M->0."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    phase = torch.exp(-1j * c * k * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, max_mode+1, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            ph = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + ph)
        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft(u_hat, n=nx))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'

def generate_burgers_data(n_samples=200, nx=64, nt=20, dt=0.005, nu=0.02):
    """Viscous Burgers: du/dt + u*du/dx = nu*d²u/dx². Mixed rev+diss."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    dx = x[1] - x[0]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, 5, (1,)).item()
            amp = torch.randn(1).item() * 0.3
            phase = torch.rand(1).item() * 2 * math.pi
            u += amp * torch.sin(ki * x + phase)

        traj = [u.clone()]
        # Semi-implicit: diffusion in spectral, advection in physical
        for t in range(nt):
            u_hat = torch.fft.rfft(u)
            # Diffusion (implicit)
            u_hat = u_hat / (1 + nu * k**2 * dt)
            u = torch.fft.irfft(u_hat, n=nx)
            # Advection (explicit, spectral derivative)
            du_dx = torch.fft.irfft(1j * k * torch.fft.rfft(u), n=nx)
            u = u - dt * u * du_dx
            traj.append(u.clone())

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'burgers'



# ============================================================================
# DeepONet building blocks
# ============================================================================

def _mlp(sizes, act=nn.GELU):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return nn.Sequential(*layers)


class DeepONetField(nn.Module):
    """DeepONet mapping a field u (B,1,N) to a field G(u) (B,1,N) on the same grid.
       branch: u sampled at the N grid sensors -> p coefficients;
       trunk:  grid coordinate -> p basis features; G(u)(y)=sum_k b_k(u) tau_k(y)."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.nx, self.p = nx, p
        self.branch = _mlp([nx, hidden, hidden, p])
        self.trunk = _mlp([1, hidden, hidden, p])
        self.bias = nn.Parameter(torch.zeros(1))
        coords = torch.linspace(0, 1, nx + 1)[:-1].unsqueeze(1)  # (N,1)
        self.register_buffer('coords', coords)

    def forward(self, u):                      # u: (B,1,N)
        b = self.branch(u.squeeze(1))          # (B,p)
        t = self.trunk(self.coords)            # (N,p)
        G = b @ t.transpose(0, 1) + self.bias  # (B,N)
        return G.unsqueeze(1)                  # (B,1,N)


class VanillaDeepONet1d(nn.Module):
    """Baseline: residual operator u_{t+1} = u_t + G(u_t) via DeepONet."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.net = DeepONetField(nx, p, hidden)

    def forward(self, u):
        return u + self.net(u)

    def predict_with_info(self, u):
        return self.forward(u), {}

    def energy_penalty(self, info, pde_type):  # for API parity (unused)
        return torch.tensor(0.0, device=next(self.parameters()).device)


class DeepONetFunctional(nn.Module):
    """Scalar functional E[u] via DeepONet: field G(u)(y) -> spatial mean -> MLP head."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.field = DeepONetField(nx, p, hidden)
        self.head = _mlp([1, 32, 1])

    def forward(self, u):                  # (B,1,N) -> (B,)
        G = self.field(u)                  # (B,1,N)
        m = G.mean(dim=(-1, -2))           # (B,)  integral functional
        return self.head(m.unsqueeze(-1)).squeeze(-1)


# ============================================================================
# GENERIC-DeepONet: DeepONet functionals + (reused) GENERIC dynamics in 1D
# ============================================================================

class GENERIC_DeepONet1d(nn.Module):
    def __init__(self, nx, p=64, hidden=128, modes_op=16,
                 degeneracy_construction=True):
        super().__init__()
        self.nx = nx
        self.degeneracy_construction = degeneracy_construction
        self.E_net = DeepONetFunctional(nx, p, hidden)
        self.S_net = DeepONetFunctional(nx, p, hidden)
        m = min(modes_op, nx // 2 + 1)
        self.m = m
        # L = i a (skew); M = |b|^2 (PSD); diagonal Fourier multipliers on low modes
        self.a = nn.Parameter(0.3 * torch.randn(m))
        self.b_r = nn.Parameter(0.3 * torch.randn(m))
        self.b_i = nn.Parameter(0.3 * torch.randn(m))
        self.residual = DeepONetField(nx, p, hidden)
        self.residual_gate = nn.Parameter(torch.tensor(-3.0))

    def _ops(self):
        return 1j * self.a, self.b_r**2 + self.b_i**2

    def _apply_ops(self, dEh, dSh):
        m = self.m
        L, M = self._ops()
        rev = torch.zeros_like(dEh)
        diss = torch.zeros_like(dSh)
        rev[:, :, :m] = L * dEh[:, :, :m]
        diss[:, :, :m] = M * dSh[:, :, :m]
        return rev, diss

    # --- degeneracy-by-construction helpers ---
    def _L_apply(self, v):
        m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (1j * self.a) * vh[:, :, :m]
        return torch.fft.irfft(out, n=self.nx, dim=-1)

    def _M_apply(self, v):
        m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (self.b_r**2 + self.b_i**2) * vh[:, :, :m]
        return torch.fft.irfft(out, n=self.nx, dim=-1)

    def _remove(self, v, w):
        return v - (self._ip(v, w) / (self._ip(w, w) + 1e-12)) * w

    def _generic_rhs(self, dEdu, dSdu):
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu)), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu)), dEdu)
        return rev, diss

    def entropy_production(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu); dudt = rev + diss
        else:
            dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
            rev, diss = self._apply_ops(dEh, dSh)
            dudt = torch.fft.irfft(rev, n=self.nx, dim=-1) + torch.fft.irfft(diss, n=self.nx, dim=-1)
            dudt = self._proj_E(dudt, dEdu); dudt = self._ensure_S(dudt, dSdu, dEdu)
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    @staticmethod
    def _ip(a, b):
        return (a * b).sum(dim=(-1, -2), keepdim=True)

    def _proj_E(self, g, dEdu):
        return g - (self._ip(g, dEdu) / (self._ip(dEdu, dEdu) + 1e-12)) * dEdu

    def _ensure_S(self, g, dSdu, dEdu):
        s_perp = dSdu - (self._ip(dSdu, dEdu) / (self._ip(dEdu, dEdu) + 1e-12)) * dEdu
        ip_sg = self._ip(dSdu, g)
        denom = self._ip(dSdu, s_perp) + 1e-12
        alpha = torch.relu(-ip_sg) / denom
        return g + alpha * s_perp

    def degeneracy_loss(self, u):
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        L, M = self._ops(); m = self.m
        L_dS = L * dSh[:, :, :m]
        M_dE = M * dEh[:, :, :m]
        return (L_dS.abs()**2).mean() + (M_dE.abs()**2).mean()

    def forward(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.degeneracy_construction:
            # degeneracy by construction: dE/dt=0, dS/dt>=0 exactly, no residual
            rev, diss = self._generic_rhs(dEdu, dSdu)
            return u + rev + diss
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        rev, diss = self._apply_ops(dEh, dSh)
        rev = torch.fft.irfft(rev, n=self.nx, dim=-1)
        diss = torch.fft.irfft(diss, n=self.nx, dim=-1)
        dudt = rev + diss
        dudt = self._proj_E(dudt, dEdu)
        dudt = self._ensure_S(dudt, dSdu, dEdu)
        gate = torch.sigmoid(self.residual_gate)
        res = self._proj_E(gate * self.residual(u_leaf), dEdu)
        return u + dudt + res

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        un = self.forward(u)
        with torch.no_grad():
            dE = (self.E_net(un) - E)
            dS = (self.S_net(un) - S)
        return un, {'dE': dE, 'dS': dS}


# ============================================================================
# Train / evaluate
# ============================================================================


# ==================== PART C: PARAM-MATCHED DEEPONET ABLATION =================
# Trains DeepONet (baseline) and GENERIC-DeepONet at TWO matched budgets (~58K
# and ~175K) on heat / advection / Burgers, so the "structure, not capacity"
# claim is controlled from both sides. Prints a matched-budget table + a
# ready-to-paste LaTeX tabular, and saves fig_deeponet_matched.{pdf,png}.
import os, math
import numpy as np
import torch
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

AB_NX      = int(os.environ.get("AB_NX", 64))
AB_NT      = int(os.environ.get("AB_NT", 20))
AB_NTRAIN  = int(os.environ.get("AB_NTRAIN", 200))
AB_NTEST   = int(os.environ.get("AB_NTEST", 50))
AB_EPOCHS  = int(os.environ.get("AB_EPOCHS", 120))
AB_BATCH   = int(os.environ.get("AB_BATCH", 16))
AB_K       = int(os.environ.get("AB_K", 2))
AB_LR      = 1e-3
AB_BUDGETS = [58_000, 175_000]
AB_PDES    = ["heat", "advection", "burgers"]
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
AB_FIGDIR  = os.environ.get("GENERIC_FNO_FIGDIR",
                            os.path.join(os.environ.get("GENERIC_FNO_BASE", "."), "figures"))
AB_GEN = {"heat": generate_heat_data, "advection": generate_advection_data,
          "burgers": generate_burgers_data}
AB_C = {"baseline": "#9467bd", "GENERIC": "#d62728"}     # DeepONet purple, GENERIC red

def _cp(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def _is_gen(m): return hasattr(m, "E_net")

def _build(kind, p, hidden, nx):
    if kind == "GENERIC":
        return GENERIC_DeepONet1d(nx, p=p, hidden=hidden, degeneracy_construction=True)
    return VanillaDeepONet1d(nx, p=p, hidden=hidden)

def _autosize(kind, target, nx):
    """Pick (p, hidden) so the model's parameter count is closest to `target`."""
    best = None
    for h in [32, 48, 64, 80, 96, 128, 160, 192, 224, 256]:
        for p in [16, 24, 32, 48, 64, 80, 96, 128]:
            try: n = _cp(_build(kind, p, h, nx))
            except Exception: continue
            if best is None or abs(n - target) < abs(best[0] - target):
                best = (n, p, h)
    return best                                          # (params, p, hidden)

def _rel(pred, tgt):
    p = pred.reshape(pred.shape[0], -1); t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.vector_norm(p - t, dim=1) /
            (torch.linalg.vector_norm(t, dim=1) + 1e-12)).mean()

def _train(model, pde):
    di, do, _ = AB_GEN[pde](n_samples=AB_NTRAIN, nx=AB_NX, nt=AB_NT)
    di = di.to(DEVICE).float(); do = do.to(DEVICE).float()          # [B, T, N]
    B, T = di.shape[0], di.shape[1]
    opt = torch.optim.Adam(model.parameters(), lr=AB_LR); model.train()
    for ep in range(AB_EPOCHS):
        perm = torch.randperm(B, device=DEVICE)
        for i in range(0, B, AB_BATCH):
            idx = perm[i:i + AB_BATCH]; di_b, do_b = di[idx], do[idx]
            t0 = int(torch.randint(0, T - AB_K + 1, (1,)).item())
            u = di_b[:, t0:t0 + 1, :]                               # [b, 1, N]
            loss = 0.0
            for k in range(AB_K):
                u = model(u)
                loss = loss + _rel(u, do_b[:, t0 + k:t0 + k + 1, :])
            (loss / AB_K).backward(); opt.step(); opt.zero_grad()
    model.eval(); return model

def _step(model, u, gen):
    if gen: return model(u).detach()              # GENERIC differentiates E,S internally
    with torch.no_grad(): return model(u)

def _eval(model, pde, n_roll=10):
    di, do, _ = AB_GEN[pde](n_samples=AB_NTEST, nx=AB_NX, nt=AB_NT)
    di = di.to(DEVICE).float(); do = do.to(DEVICE).float()
    gen = _is_gen(model); u = di[:, 0:1, :]; errs = []
    for t in range(min(n_roll, di.shape[1])):
        u = _step(model, u, gen); tgt = do[:, t:t + 1, :]
        errs.append((torch.linalg.vector_norm((u - tgt).reshape(u.shape[0], -1), dim=1) /
                     (torch.linalg.vector_norm(tgt.reshape(tgt.shape[0], -1), dim=1) + 1e-12)).mean().item())
    return float(np.mean(errs))

def run_deeponet_matched(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    print(f"Param-matched DeepONet ablation  nx={AB_NX} nt={AB_NT} epochs={AB_EPOCHS} device={DEVICE}")
    sizes = {}
    for b in AB_BUDGETS:
        for kind in ["baseline", "GENERIC"]:
            n, p, h = _autosize(kind, b, AB_NX)
            sizes[(kind, b)] = (n, p, h)
            print(f"  target {b//1000}K  {kind:8s}: {n:6d} params  (p={p}, hidden={h})")
    results = {}
    for pde in AB_PDES:
        for b in AB_BUDGETS:
            for kind in ["baseline", "GENERIC"]:
                n, p, h = sizes[(kind, b)]
                model = _build(kind, p, h, AB_NX).to(DEVICE)
                _train(model, pde)
                results[(pde, kind, b)] = _eval(model, pde)
                print(f"    {pde:9s} {kind:8s} @{b//1000}K ({n} params): rollout L2 = {results[(pde,kind,b)]:.3f}")
    _report(results, sizes); _plot(results, sizes)
    return results, sizes

def _report(results, sizes):
    bsmall, blarge = AB_BUDGETS
    print("\n==== matched-budget rollout L2 (lower is better; * = GENERIC wins) ====")
    wins = tot = 0
    for pde in AB_PDES:
        cells = []
        for b in AB_BUDGETS:
            bl = results[(pde, "baseline", b)]; gn = results[(pde, "GENERIC", b)]
            tot += 1; wins += int(gn < bl)
            cells.append(f"base {bl:.3f}  gen {gn:.3f}{'*' if gn < bl else ' '}")
        print(f"  {pde:9s} | {bsmall//1000}K: {cells[0]} | {blarge//1000}K: {cells[1]}")
    print(f"  GENERIC lower at {wins}/{tot} matched-budget comparisons")
    # ready-to-paste LaTeX
    ps = {k: sizes[k][0] for k in sizes}
    print("\n% ---- LaTeX (Appendix app:donmatch) ----")
    print(r"\begin{tabular}{lcccc}")
    print(r"\toprule")
    print(r" & \multicolumn{2}{c}{$\approx$%dK params} & \multicolumn{2}{c}{$\approx$%dK params} \\"
          % (bsmall // 1000, blarge // 1000))
    print(r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}")
    print(r"PDE & DeepONet & GENERIC & DeepONet & GENERIC \\")
    print(r"\midrule")
    for pde in AB_PDES:
        vals = []
        for b in AB_BUDGETS:
            bl = results[(pde, "baseline", b)]; gn = results[(pde, "GENERIC", b)]
            vals += [f"{bl:.3f}", (r"\textbf{%.3f}" % gn) if gn < bl else f"{gn:.3f}"]
        print(f"{pde.capitalize()} & " + " & ".join(vals) + r" \\")
    print(r"\bottomrule")
    print(r"\end{tabular}")

def _plot(results, sizes):
    os.makedirs(AB_FIGDIR, exist_ok=True)
    fig, axes = plt.subplots(1, len(AB_BUDGETS), figsize=(5.2 * len(AB_BUDGETS), 3.6), squeeze=False)
    x = np.arange(len(AB_PDES)); w = 0.36
    for j, b in enumerate(AB_BUDGETS):
        ax = axes[0][j]
        bl = [results[(p, "baseline", b)] for p in AB_PDES]
        gn = [results[(p, "GENERIC", b)] for p in AB_PDES]
        ax.bar(x - w/2, bl, w, color=AB_C["baseline"], label="DeepONet")
        ax.bar(x + w/2, gn, w, color=AB_C["GENERIC"], label="GENERIC-DeepONet")
        ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels([p.capitalize() for p in AB_PDES])
        ax.set_ylabel("rollout $L^2$ (log)")
        ax.set_title(f"$\\approx${b//1000}K params "
                     f"(base {sizes[('baseline',b)][0]//1000}K, gen {sizes[('GENERIC',b)][0]//1000}K)")
        ax.grid(axis="y", ls=":", alpha=0.5)
        if j == 0: ax.legend(fontsize=8)
    fig.suptitle("Param-matched DeepONet: GENERIC wins at equal budget (structure, not capacity)",
                 y=1.03, fontsize=11)
    fig.tight_layout()
    done = []
    for ext in ("pdf", "png"):
        try:
            fig.savefig(os.path.join(AB_FIGDIR, f"fig_deeponet_matched.{ext}"), bbox_inches="tight")
            done.append(ext)
        except Exception as e:
            print(f"  [WARN] save {ext}: {e}")
    plt.close(fig)
    print(f"  saved fig_deeponet_matched: {'+'.join(done)} -> {AB_FIGDIR}")

if __name__ == "__main__":
    run_deeponet_matched()

Param-matched DeepONet ablation  nx=64 nt=20 epochs=120 device=cuda
  target 58K  baseline:  58113 params  (p=64, hidden=128)
  target 58K  GENERIC :  56790 params  (p=48, hidden=64)
  target 175K  baseline: 173409 params  (p=128, hidden=224)
  target 175K  GENERIC : 174582 params  (p=64, hidden=128)
    heat      baseline @58K (58113 params): rollout L2 = 0.055
    heat      GENERIC  @58K (56790 params): rollout L2 = 0.004
    heat      baseline @175K (173409 params): rollout L2 = 0.052
    heat      GENERIC  @175K (174582 params): rollout L2 = 0.004
    advection baseline @58K (58113 params): rollout L2 = 0.209
    advection GENERIC  @58K (56790 params): rollout L2 = 0.010
    advection baseline @175K (173409 params): rollout L2 = 0.190
    advection GENERIC  @175K (174582 params): rollout L2 = 0.009
    burgers   baseline @58K (58113 params): rollout L2 = 0.024
    burgers   GENERIC  @58K (56790 params): rollout L2 = 0.009
    burgers   baseline @175K (173409 params): rollout L2 = 0